# Baselines - Part 3: B1 and B2 on fold I (3 seeds)

Trains the TESSERA 2019 neural baselines on fold I at 100% labels across seeds 0, 1, and 2:
- B1 (`base_mlp-t19`): per-pixel MLP (`1x1` convolutions, no spatial context)
- B2 (`base_unet-t19-w32`): light UNet with width `32-64-128` (485,267 parameters, kept fixed below the 1.1M parameters of U-TAE since validation mIoU plateaus around step 2,000 in Part 2)

We then aggregate test results over the 496 test patches with patch-bootstrap confidence intervals for B0 (linear head), B1, and B2 across all valid pixels, parcel boundaries, and parcel cores, along with paired differences (`B2 - B1`, `B2 - B0`, `B1 - B0`).

## 1. Environment and paths

In [ ]:
import json, os, shutil, platform, sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
print("Python", platform.python_version(), "| torch", torch.__version__, f"| {os.cpu_count()} CPUs")
assert torch.cuda.is_available(), "no GPU: Runtime > Change runtime type > T4 GPU, then run again"
print("GPU:", torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

from google.colab import drive
drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive/pj_vmi")
REPO_URL = "https://github.com/ColdCastlee/ProjectVMI.git"
BRANCH = "main"  # or the branch being tested
CODE = Path("/content/code")
PROC_DRIVE = DRIVE / "processed"
PROC = Path("/content/processed")
RUNS, RUNS_CSV = DRIVE / "runs", DRIVE / "results" / "runs.csv"
OUT = DRIVE / "results" / "baselines"
B1 = "configs/baselines/b1_mlp_t19.yaml"
B2 = "configs/baselines/b2_unet_t19.yaml"

## 2. Repository and unit tests

In [ ]:
if (CODE / ".git").exists():
    !git -C "{CODE}" fetch -q origin {BRANCH} && git -C "{CODE}" checkout -q {BRANCH} && git -C "{CODE}" pull -q --ff-only origin {BRANCH}
else:
    !git clone -q -b {BRANCH} {REPO_URL} "{CODE}"
%cd {CODE}
!git log -1 --format="commit %h  %ad  %s" --date=short
!pip install -q -r requirements.txt
!python -m pytest -q tests
sys.path.insert(0, str(CODE))

## 3. Copy TESSERA 2019 data to local disk

In [ ]:
PROC.mkdir(parents=True, exist_ok=True)
for f in ["meta.csv", "splits.json", "labels.npy", "instances.npy", "valid_mask.npy",
          "tessera_2019_q.npy", "tessera_2019_scale.npy"]:
    if not (PROC / f).exists() or (PROC / f).stat().st_size != (PROC_DRIVE / f).stat().st_size:
        shutil.copy(PROC_DRIVE / f, PROC / f)
    print(f"{f:24s} {(PROC / f).stat().st_size / 1e9:6.2f} GB  ok={(PROC / f).stat().st_size == (PROC_DRIVE / f).stat().st_size}")

## 4. Train B2 (TESSERA 2019 + light UNet, seeds 0, 1, 2)
Seed 0 from Part 2 is already saved on Drive, so the script only computes its per-patch validation confusion matrices (`cm_val_per_patch.npz`) from `best.pt` and trains seeds 1 and 2.

In [ ]:
!python scripts/train_head.py --config {B2} --processed {PROC} --runs-dir {RUNS} --runs-csv {RUNS_CSV} --seeds 0 1 2

## 5. Train B1 (TESSERA 2019 + pixel MLP, seeds 0, 1, 2)
Train the per-pixel MLP (`1x1` convolutions without spatial context) across seeds 0, 1, and 2 using the same optimizer budget and loss as B2.

In [ ]:
!python scripts/train_head.py --config {B1} --processed {PROC} --runs-dir {RUNS} --runs-csv {RUNS_CSV} --seeds 0 1 2

## 6. Test set summary and paired bootstrap comparisons
Aggregate test mIoU over the 3 seeds with patch-bootstrap confidence intervals across all valid pixels, parcel boundaries, and parcel cores, and compute paired differences (`B2 - B1`, `B2 - B0`, `B1 - B0`). Comparing B2 against B1 measures the gain from spatial context on identical TESSERA 2019 inputs.

In [ ]:
from IPython.display import Image, display
from scripts.summarize_runs import summarize_runs

B0, B1_TAG, B2_TAG = "lin_t19", "base_mlp-t19", "base_unet-t19-w32"
pd.set_option("display.width", 220)
per_seed, summary, deltas = summarize_runs(
    RUNS, OUT, name="baselines_foldI", variants=[B0, B1_TAG, B2_TAG],
    pairs=[f"{B2_TAG}:{B1_TAG}", f"{B2_TAG}:{B0}", f"{B1_TAG}:{B0}"], split="I", label_frac=1.0, seeds=(0, 1, 2))
print(summary[["variant", "seeds", "miou_mean", "miou_std", "ci_low", "ci_high", "boundary_mean", "core_mean",
               "val_miou_mean", "n_params", "train_seconds_mean"]].round(4).to_string(index=False))
print("\npaired bootstrap, delta = mIoU(variant) - mIoU(versus), averaged over seeds:")
print(deltas.round(4).to_string(index=False))
display(Image(str(OUT / "baselines_foldI_per_class.png")))